In [ ]:
%%capture
import os
from pathlib import Path
import pandas as pd

from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
reports_folder = Path(os.environ["INTECOMM_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)


In [ ]:
from intecomm_analytics.dataframes import get_df_main_1858
from edc_constants.constants import NO, YES
from intecomm_analytics.dataframes import treatment_arm_labels as treatment_arm
from intecomm_rando.constants import FACILITY_ARM, COMMUNITY_ARM

In [ ]:
narrative = []

# boudaries for first measurement
baseline_lower_bound = -180
baseline_upper_bound = 31

# boudaries for last measurement
endline_lower_bound = 182

# boundaries for diagnosis
days_since_dx = 180

In [ ]:
# get 1858
df_main_original = get_df_main_1858(None)
df_main = df_main_original.copy()

In [ ]:
df_main.country.value_counts()

In [ ]:
msg = f"There are {len(df_main[(df_main.hiv_scr==1)])}/{len(df_main)} subjects with HIV reported at screening"
narrative.append(msg)
print(msg)


In [ ]:
msg = f"There are {len( df_main[(df_main.hiv==1)])}/{len(df_main)} subjects with HIV confirmed at baseline and diagnosed at least {days_since_dx} days before baseline"
narrative.append(msg)
print(msg)


In [ ]:
msg = f"There are {len(df_main[(df_main.hiv==1) & (df_main.dm==0) & (df_main.htn==0)])}/{len(df_main)} HIV(+) only subjects confirmed at baseline and diagnosed at least {days_since_dx} days before baseline"
narrative.append(msg)
print(msg)

In [ ]:
print("\n".join(narrative))

In [ ]:
len(df_main[(df_main.hiv==0) & (df_main.dm==1)])

In [ ]:
def get_cells_for_continuous_var(df)->list[str]:
    return [
        f"{int(df['count'])}",
        f"{df['mean']:.2f}({df['std']:.2f})",
        f"{df['50%']:.2f}({df['min']:.2f}–{df['max']:.2f})"
    ]

def get_cells_for_yes_no(df:pd.DataFrame, col:str, arm:str|None=None)->list[str]:
    if arm:
        n = len(df[(df['assignment']==arm) & (df[col].notna())])
        counts = df[(df['assignment'] == arm) & (df[col].notna())][col].value_counts()
        percentages = df[(df['assignment'] == arm) & (df[col].notna())][col].value_counts(normalize=True) * 100
    else:
        n = len(df[(df[col].notna())])
        counts = df[(df[col].notna())][col].value_counts()
        percentages = df[(df[col].notna())][col].value_counts(normalize=True) * 100
    return [
        n,
        f"{counts.get(YES, 0)} ({percentages.get(YES, 0):.1f}%)",
        f"{counts.get(NO, 0)} ({percentages.get(NO, 0):.1f}%)"]

def get_cells_for_yes_no_missing(df:pd.DataFrame, col:str, arm:str|None=None)->list[str]:
    if arm:
        n = len(df[(df['assignment']==arm) & (df[col].notna())])
        counts = df[(df['assignment'] == arm) & (df[col].notna())][col].value_counts()
        percentages = df[(df['assignment'] == arm) & (df[col].notna())][col].value_counts(normalize=True) * 100
    else:
        n = len(df[(df[col].notna())])
        counts = df[(df[col].notna())][col].value_counts()
        percentages = df[(df[col].notna())][col].value_counts(normalize=True) * 100
    return [
        n,
        f"{counts.get(YES, 0)} ({percentages.get(YES, 0):.1f}%)",
        f"{counts.get(NO, 0)} ({percentages.get(NO, 0):.1f}%)",
        f"{counts.get('Missing', 0)} ({percentages.get('Missing', 0):.1f}%)"]


def get_formatted_rows_vl(df, col_baseline:str|None=None, col_endline:str|None=None):
    """Returns 5 columns"""

    df_base = df.copy()
    baseline_a = df_base[df_base['assignment'] == 'a'][col_baseline].describe()
    baseline_b = df_base[df_base['assignment'] == 'b'][col_baseline].describe()
    baseline_all = df_base[col_baseline].describe()

    df_end = df[(df["onstudy_days"] >= 182)].copy()
    endline_a = df_end[df_end['assignment'] == 'a'][col_endline].describe()
    endline_b = df_end[df_end['assignment'] == 'b'][col_endline].describe()
    endline_all = df_end[col_endline].describe()

    return  {
        'Timepoint': ['Baseline', '', '', 'Endline', '', ''],
        'Statistics': ['n', 'Mean(sd)', 'Median(min-max)','n', 'Mean(sd)', 'Median(min-max)'],
        treatment_arm[COMMUNITY_ARM]: [
            *get_cells_for_continuous_var(baseline_a),
            *get_cells_for_continuous_var(endline_a),
        ],
        treatment_arm[FACILITY_ARM]: [
            *get_cells_for_continuous_var(baseline_b),
            *get_cells_for_continuous_var(endline_b),
        ],
        'All': [
            *get_cells_for_continuous_var(baseline_all),
            *get_cells_for_continuous_var(endline_all),
        ],
    }

def get_formatted_rows_yes_no(df_base:pd.DataFrame,df_end:pd.DataFrame, baseline_col:str, endline_col:str, missing:bool|None=None):
    """Returns 5 columns"""
    rows = {}
    if missing:
        func = get_cells_for_yes_no_missing
        rows.update({
            'Timepoint': ['Baseline', '', '','', 'Endline', '', '',''],
            'Statistics': ['n', 'Yes', 'No', "Missing", 'n', 'Yes','No', "Missing"]})
    else:
        func = get_cells_for_yes_no
        rows.update({
            'Timepoint': ['Baseline', '', '', 'Endline', '', ''],
            'Statistics': ['n', 'Yes', 'No', 'n', 'Yes', 'No',]})
    rows.update({
        treatment_arm[COMMUNITY_ARM]: [
        *func(df_base, baseline_col, arm="a"),
        *func(df_end, endline_col, arm="a"),
        ],
        treatment_arm[FACILITY_ARM]: [
            *func(df_base, baseline_col, arm="b"),
            *func(df_end, endline_col, arm="b"),
        ],
        'All': [
            *func(df_base, baseline_col),
            *func(df_end, endline_col),
        ],
        })
    return rows

In [ ]:
# create df_main filtered by condition
df_hiv = df_main[(df_main.htn==0) & (df_main.dm==0) & (df_main.hiv==1)][["subject_identifier", "vl_baseline", "vl_endline", "ncd", "hiv", "dm", "htn", "assignment", "country", "onstudy_days"]].copy()
df_hiv.reset_index(inplace=True, drop=True)

In [ ]:
df_hiv

In [ ]:
print(f"{df_hiv[df_hiv.vl_baseline.notna()]["subject_identifier"].count()} first results")
print(f"{df_hiv[df_hiv.vl_endline.notna()]["subject_identifier"].count()} last results")
print(f"{df_hiv[(df_hiv.vl_baseline.notna()) & (df_hiv.vl_endline.notna())]["subject_identifier"].count()} first and last results")

In [ ]:
df_hiv["vl_baseline"] = df_hiv["vl_baseline"].astype(float)
df_hiv["vl_endline"] = df_hiv["vl_endline"].astype(float)
path = analysis_folder / "df_hiv.csv"
df_hiv.to_csv(path, index=False)


In [ ]:
df_hiv

In [ ]:
# All
vl_table = {'Condition': ['HIV only', '', '', '', '', '']}
vl_table.update({
    'Parameter': ['VL (copies/mL)', '', '', '', '', ''],
    **get_formatted_rows_vl(df_hiv, "vl_baseline", "vl_endline")
})


In [ ]:
table_hiv_df = pd.DataFrame(vl_table)
table_hiv_df


In [ ]:
vl_table = {'Condition': ['HIV only', '', '', '', '', '']}
vl_table.update({
    'Parameter': ['VL (log10)', '', '', '', '', ''],
    **get_formatted_rows_vl(df_hiv, "vl_baseline_log10", "vl_endline_log10")
})
table_hiv_log_df = pd.DataFrame(vl_table)
table_hiv_log_df

In [ ]:
table_hiv_df = pd.concat([table_hiv_df, table_hiv_log_df])

In [ ]:
path = analysis_folder / 'vl.csv'
table_hiv_df.to_csv(path_or_buf=path, index=False)
